# Lezione 9 — VQE: fondamenti

## Obiettivi di apprendimento

- Scomporre un Hamiltoniano in termini di Pauli
- Applicare il principio variazionale per stimare un'energia minima
- Ottimizzare un ansatz con il gradiente e con Nelder-Mead

In questa lezione conosci uno degli algoritmi quantistici variazionali con applicazioni reali nella chimica quantistica: il Risolutore Variazionale Quantistico di Autovalori (VQE, dall'inglese Variational Quantum Eigensolver). Impari a costruire un Hamiltoniano a partire da matrici di Pauli, a misurarlo con un circuito parametrizzato e a ottimizzare quel circuito per trovare l'energia dello stato fondamentale di un sistema quantistico.

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum scipy

## Perché è importante

Uno degli usi più promettenti del calcolo quantistico è la simulazione di sistemi fisici e molecole, un compito che un computer classico svolge molto male perché gli stati quantistici crescono in modo esponenziale (ricorda la lezione 3). VQE è l'algoritmo di riferimento per calcolare l'energia dello stato fondamentale (il livello energetico più basso possibile) di un sistema quantistico, ed è alla base di applicazioni reali nella chimica quantistica, come il calcolo dell'energia di una molecola di idrogeno.

## Un Hamiltoniano è una "ricetta" dell'energia

In fisica quantistica, l'**Hamiltoniano** di un sistema è l'operatore che descrive la sua energia totale. Puoi scomporre qualsiasi Hamiltoniano come somma pesata di prodotti tensoriali delle matrici di Pauli (I, X, Y, Z, che conosci già dalle lezioni precedenti). Per esempio, un Hamiltoniano di un solo qubit si scrive come `H = a·I + b·X + c·Y + d·Z`, dove a, b, c e d sono numeri reali che dipendono dal sistema fisico che vuoi descrivere. L'obiettivo di VQE è trovare il valore minimo possibile del valore di attesa di quell'Hamiltoniano: per definizione, è l'energia dello stato fondamentale.

## Il principio variazionale: indovina e migliora

VQE si basa sul **principio variazionale**: per qualsiasi stato quantistico tu provi, il valore di attesa dell'Hamiltoniano su quello stato non scende mai sotto l'energia reale dello stato fondamentale. Questo trasforma la ricerca dell'energia minima in un problema di ottimizzazione: prepari un circuito con parametri regolabili (l'"ansatz", la tua migliore congettura su come dovrebbe apparire lo stato fondamentale), misuri il valore di attesa dell'Hamiltoniano e regoli i parametri in modo che quel valore scenda il più possibile. Più lo abbassi, più ti avvicini all'energia reale.

## Misurare un Hamiltoniano con più termini

Un Hamiltoniano su più qubit, come `H = a·(Z₀+Z₁) + b·(Z₂+Z₃)`, si misura termine per termine: per ciascuno, misuri il circuito nella base corrispondente (ruotando prima della misura se il termine contiene X o Y, perché puoi misurare direttamente solo nella base Z), moltiplichi per il suo coefficiente e sommi tutti i risultati. TFQ lo fa in modo trasparente: gli passi la lista dei termini come operatore di lettura di un livello `PQC`, e quel livello si occupa internamente delle misure e della somma.

## Ottimizzatori classici: gradiente contro scatola nera

Hai due modi per ottimizzare i parametri dell'ansatz:

- **Basata sul gradiente**: usi `tf.GradientTape` e un ottimizzatore come Adam, come nelle lezioni precedenti, ogni volta che puoi calcolare il gradiente del circuito (con `ParameterShift`, per esempio)
- **Ottimizzazione a scatola nera**: tratti il circuito come una funzione che puoi solo valutare (senza gradiente), e usi algoritmi come `scipy.optimize.minimize` con il metodo Nelder-Mead, che prova diverse combinazioni di parametri e converge senza bisogno di derivate

Entrambi gli approcci sono validi; nella prossima lezione vedrai anche l'algoritmo Rotosolve, progettato specificamente per i circuiti quantistici parametrizzati.

## Esempio 1 — Hamiltoniano di un qubit con Nelder-Mead

Parti dal caso più semplice possibile: un solo qubit, un ansatz con una sola rotazione `ry` e un Hamiltoniano `H = 1·I + 2·Z`, la cui energia fondamentale teorica è 1 − 2 = −1. Poiché non calcoli gradienti, ottimizzi con `scipy.optimize.minimize` e il metodo Nelder-Mead, che ha solo bisogno di poter valutare l'energia per ogni valore dell'angolo. Alla fine, scandisci l'angolo tra 0 e 2π per visualizzare l'intero paesaggio energetico e verificare che il minimo trovato dall'ottimizzatore coincida con il minimo reale della curva.

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

qubit = cirq.GridQubit(0, 0)
angulo = sympy.symbols("angulo")

# Ansatz: una única rotación Y entrenable
circuito_vqe = cirq.Circuit(cirq.ry(angulo).on(qubit))

# Hamiltoniano H = 1*I + 2*Z, cuya energía fundamental teórica es 1 - 2 = -1
hamiltoniano = 1 + 2 * cirq.Z(qubit)

entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
inicializador = tf.keras.initializers.RandomUniform(0, 2 * np.pi)
salida = tfq.layers.PQC(circuito_vqe, hamiltoniano,
                         differentiator=tfq.differentiators.CentralDifference(),
                         initializer=inicializador)(capa_entrada)
modelo_vqe = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)


def energia(parametros):
    modelo_vqe.set_weights(np.array([parametros]))
    return modelo_vqe(entrada_vacia).numpy()[0][0]


resultado = minimize(energia, np.random.uniform(0, 2 * np.pi, 1), method="Nelder-Mead")
print("Energía mínima encontrada:", resultado["fun"])
print("Ángulo óptimo:", resultado["x"])

# Visualizas el paisaje de energía completo, barriendo el ángulo entre 0 y 2*pi
angulos = np.linspace(0, 2 * np.pi, 100)
energias = [energia([a]) for a in angulos]
plt.plot(angulos, energias)
plt.xlabel("Ángulo del ansatz")
plt.ylabel("Energía")
plt.title("VQE — paisaje de energía de un único parámetro")
plt.show()

## Esempio 2 — Hamiltoniano di quattro qubit con Adam

Ora passi al livello successivo: un ansatz a quattro qubit con due parametri addestrabili (`rx` e `ry` su ciascun qubit, più una porta `H` e due `CNOT` che mettono in entanglement i qubit) e un Hamiltoniano con due termini indipendenti, `H = 1·(Z₀+Z₁) + 1·(Z₂+Z₃)`. Qui puoi calcolare i gradienti, quindi addestri con `tf.GradientTape` e l'ottimizzatore Adam, esattamente come addestreresti una qualsiasi rete neurale classica. Osserva come l'energia scende progressivamente lungo le cento iterazioni di addestramento.

In [ ]:
def construir_ansatz(qubits, parametros):
    circuito = cirq.Circuit()
    for i, qubit in enumerate(qubits):
        circuito.append(cirq.rx(parametros[0]).on(qubit))
        circuito.append(cirq.ry(parametros[1]).on(qubit))
    circuito.append(cirq.H(qubits[1]))
    circuito.append(cirq.CNOT(qubits[0], qubits[1]))
    circuito.append(cirq.CNOT(qubits[2], qubits[3]))
    return circuito


qubits = [cirq.GridQubit(0, i) for i in range(4)]
parametros = sympy.symbols("theta0:2")
circuito_vqe_multi = construir_ansatz(qubits, parametros)

# H = 1*(Z0 + Z1) + 1*(Z2 + Z3)
hamiltoniano_multi = (cirq.Z(qubits[0]) + cirq.Z(qubits[1])) + (cirq.Z(qubits[2]) + cirq.Z(qubits[3]))

entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_vqe_multi, hamiltoniano_multi)(capa_entrada)
modelo_multi = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)

optimizador = tf.keras.optimizers.Adam(learning_rate=0.1)
historial_energia = []
for paso in range(100):
    with tf.GradientTape() as cinta:
        energia_actual = modelo_multi(entrada_vacia)
    gradientes = cinta.gradient(energia_actual, modelo_multi.trainable_variables)
    optimizador.apply_gradients(zip(gradientes, modelo_multi.trainable_variables))
    historial_energia.append(energia_actual.numpy()[0][0])

plt.plot(historial_energia)
plt.xlabel("Iteración")
plt.ylabel("Energía")
plt.title("VQE multi-qubit — convergencia por gradiente")
plt.show()

## Riepilogo dei punti chiave

- Qualsiasi Hamiltoniano si scompone in una somma di prodotti di matrici di Pauli (I, X, Y, Z); VQE cerca di minimizzarne il valore di attesa
- Il principio variazionale garantisce che quel minimo, abbassato il più possibile, si avvicina all'energia reale dello stato fondamentale
- Un Hamiltoniano con più termini si misura termine per termine, e TFQ somma i risultati automaticamente quando glieli passi come lista al `PQC`
- Puoi ottimizzare con il gradiente (Adam + `ParameterShift`/`CentralDifference`) oppure con ottimizzatori a scatola nera come Nelder-Mead, senza bisogno di gradienti